Part 1 – Environment Setup

In [ ]:
# ============================================================
# Part 1: Environment Setup & Net2Brain Installation
# ============================================================

# Install Python 3.10 and virtual environment tools
!apt-get update -y
!apt-get install python3.10 python3.10-dev python3.10-venv -y

# Create a virtual environment with Python 3.10
!python3.10 -m venv net2brain_env

# Upgrade pip, setuptools, wheel inside the venv
!./net2brain_env/bin/pip install --upgrade pip setuptools wheel

# Install torch and torchvision
!./net2brain_env/bin/pip install torch torchvision

# Install net2brain from GitHub
!./net2brain_env/bin/pip install git+https://github.com/cvai-roig-lab/Net2Brain.git

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64

Part 2 – Basic Inspection of Net2Brain

In [ ]:
# ============================================================
# Part 2: Net2Brain Basic Inspection
# ============================================================

import os
os.environ['MPLBACKEND'] = 'Agg'

# Check torch
!./net2brain_env/bin/python -c "import torch; print('torch version:', torch.__version__)"
# Check net2brain
!./net2brain_env/bin/python -c "import net2brain; print('net2brain installed')"
# List datasets
list_script = """
from net2brain.utils.download_datasets import list_available_datasets
print('Available datasets:', list_available_datasets())
"""
with open('/tmp/list.py', 'w') as f:
    f.write(list_script)
!./net2brain_env/bin/python /tmp/list.py
print("\n✅ Part 2 complete.")

torch version: 2.13.0+cu130
net2brain installed
Available datasets: ['DatasetBonnerPNAS2017', 'Dataset78images', 'Workhsop_Harry_Potter_Cognition', 'Dataset92images', 'DatasetBoldMoments', 'WorkshopCuttingGardens', 'Tutorial_LE_Results', 'DatasetAlgonauts_NSD_Shared', 'DatasetThings_fMRI', 'DatasetAlgonauts_NSD']

✅ Part 2 complete.


Part 3 – RSA Pipeline for Table Generation

In [ ]:
%%writefile run_rsa_92images.py

# ============================================================
# RSA Analysis on 92images Dataset (Algonauts 2019, Training Set A)
# Contains 92 natural images, fMRI responses, and precomputed RDMs.
# Generates a table with rows: EVC, IT (the two official ROIs)
# and columns: DNN layers, values = R².
# ============================================================

import os
os.environ['MPLBACKEND'] = 'Agg'

import torch
import numpy as np
import pandas as pd
import re

from net2brain.utils.download_datasets import Dataset92images
from net2brain.feature_extraction import FeatureExtractor
from net2brain.rdm_creation import RDMCreator
from net2brain.evaluations.rsa import RSA

# ---------- Helper functions ----------
def extract_roi_short(roi_str):
    """Extract short ROI name from full path string."""
    match = re.search(r'fmri_(\w+)_RDMs', roi_str)
    return match.group(1) if match else roi_str

def extract_layer_short(layer_str):
    """Extract short layer name from RDM filename."""
    match = re.search(r'RDM_(\w+)\.npz', layer_str)
    return match.group(1) if match else layer_str

def natural_sort_key(s):
    """Sort keys with numbers in natural order (e.g., layer1, layer2, ...)."""
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', s)]

# ---------- 1. GPU setup ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
SAVE_ROOT = './'

# ---------- 2. Load 92images dataset ----------
print("\n[Dataset] Loading 92images dataset (92 images, training set A)...")
paths = Dataset92images().load_dataset()
print(f"Available keys: {list(paths.keys())}")
stimuli_path = paths["stimuli_path"]
brain_rdm_path = paths["roi_path"]
print(f"Stimuli path: {stimuli_path}")
print(f"Brain RDM path: {brain_rdm_path}")

# Define the two official ROIs from the Algonauts 2019 dataset
TARGET_ROIS = ['EVC', 'IT']
print(f"Target ROIs: {TARGET_ROIS}")

# ---------- 3. Define models ----------
model_configs = [
    {'name': 'ResNet18', 'netset': 'Standard'},
    {'name': 'ResNet50', 'netset': 'Standard'},
    {'name': 'ResNet152', 'netset': 'Standard'},
]
print(f"\nModels to evaluate: {[m['name'] for m in model_configs]}")

# ---------- 4. Main loop ----------
for config in model_configs:
    model_name = config['name']
    netset = config['netset']

    print(f"\n{'='*70}")
    print(f"Processing model: {model_name} (netset: {netset})")
    print('='*70)

    # 4a. Feature extraction
    print(f"[{model_name}] Extracting features from images...")
    feature_extractor = FeatureExtractor(
        model=model_name,
        netset=netset,
        device=device
    )
    model_feat_path = os.path.join(SAVE_ROOT, f'{model_name}_92images_feat')
    feature_extractor.extract(
        data_path=stimuli_path,
        save_path=model_feat_path,
        # Uncomment to reduce memory usage:
        # dim_reduction='srp', n_components=200
    )
    print(f"[{model_name}] Feature extraction complete.")

    # 4b. Build model RDM
    print(f"[{model_name}] Building model RDM...")
    rdm_creator = RDMCreator(verbose=True, device=device)
    model_rdm_path = os.path.join(SAVE_ROOT, f'{model_name}_92images_rdm')
    rdm_creator.create_rdms(
        feature_path=model_feat_path,
        save_path=model_rdm_path,
        save_format='npz'
    )
    print(f"[{model_name}] Model RDM created.")

    # 4c. Run RSA
    print(f"[{model_name}] Running RSA evaluation...")
    rsa = RSA(
        model_rdms_path=model_rdm_path,
        brain_rdms_path=brain_rdm_path,
        model_name=model_name
    )
    results_df = rsa.evaluate()
    print(f"[{model_name}] Results shape: {results_df.shape}")

    # Save full results
    full_csv = f'{model_name}_92images_rsa_full.csv'
    results_df.to_csv(full_csv, index=False)
    print(f"[{model_name}] Full results saved to {full_csv}")

    # ============================================================
    # 4d. Generate table: rows = EVC, IT; columns = layers; values = R²
    # ============================================================
    results_df['ROI_short'] = results_df['ROI'].apply(extract_roi_short)
    results_df['Layer_short'] = results_df['Layer'].apply(extract_layer_short)

    # Filter to only the two target ROIs
    filtered = results_df[results_df['ROI_short'].isin(TARGET_ROIS)]

    # Build pivot table
    if not filtered.empty:
        pivot = filtered.pivot_table(
            index='ROI_short',
            columns='Layer_short',
            values='R2',
            aggfunc='mean'
        )
    else:
        # If no data found, create an empty DataFrame with the target rows
        pivot = pd.DataFrame(index=TARGET_ROIS)

    # Enforce row order: EVC first, then IT
    pivot = pivot.reindex(TARGET_ROIS)

    # Sort columns naturally
    cols = sorted(pivot.columns, key=natural_sort_key) if not pivot.empty else []
    if cols:
        pivot = pivot[cols]

    # Replace NaN with "N/A"
    pivot = pivot.fillna("N/A")

    # Save the table
    table_csv = f'{model_name}_92images_EVC_IT_R2_table.csv'
    pivot.to_csv(table_csv)
    print(f"[{model_name}] Table saved to {table_csv}")
    print(f"\nPreview of {model_name} table:")
    print(pivot.to_string())

print("\nAll tables generated successfully.")

Writing run_rsa_92images.py


Part 4 – Execute the Script

In [ ]:
# ============================================================
# Part 4: Execute the RSA Analysis Script
# ============================================================

!./net2brain_env/bin/python run_rsa_92images.py

Using device: cuda
GPU name: NVIDIA L4

[Dataset] Loading 92images dataset (92 images, training set A)...

Extracting archive...
Unwrapped nested folder: 92images
Successfully extracted
Available keys: ['stimuli_path', 'roi_path']
Stimuli path: /content/92images/stimuli_data
Brain RDM path: /content/92images/brain_data
Target ROIs: ['EVC', 'IT']

Models to evaluate: ['ResNet18', 'ResNet50', 'ResNet152']

Processing model: ResNet18 (netset: Standard)
[ResNet18] Extracting features from images...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


Consolidating data per layer...

[ResNet18] Feature extraction complete.
[ResNet18] Building model RDM...
/content/net2brain_env/lib/python3.10/site-packages/net2brain/rdm/feature_iterator.py:388: UserWarning: Keyword arguments provided for dimensionality reduction only supported for NPZ_SEPARATE, and will have no effect for the provided format FeatureFormat.NPZ_